In [ ]:
import polars as pl
from project_paths import paths

In [ ]:
df = pl.read_parquet(paths.processed_data / "unified_aims_eir_bgs.parquet")
print(df.shape)
df.head()

In [ ]:
df = df.with_columns(
    pl.when(pl.col("eir__condition_grade") == 0).then(None).otherwise(pl.col("eir__condition_grade")).alias("_grade")
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

columns = [
    "aims__asset_length",
    "aims__toe_level",
    "aims__current_sop",
    "aims__design_sop",
    "aims__design_ucl",
    "aims__actual_dcl",
]

fig, axes = plt.subplots(6, 1, figsize=(12,18))

for index, column in enumerate(columns):
    ax = axes[index]
    ax.set_title(column)
    sns.histplot(df.get_column(column), bins=50, stat="count", ax=ax)

plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

columns = [
    "aims__asset_length",
    "aims__toe_level",
    "aims__current_sop",
    "aims__design_sop",
    "aims__design_ucl",
    "aims__actual_dcl",
]

fig, axes = plt.subplots(6, 1, figsize=(12,18))

for index, column in enumerate(columns):
    ax = axes[index]
    ax.set_title(column)

    col = df.get_column(column)

    skew = stats.skew(col)
    kurtosis = stats.kurtosis(col)
    min = col.min()
    median = col.median()
    max = col.max()

    summary = pl.DataFrame({"skew": skew, "kurtosis": kurtosis, "min": min, "median": median, "max":max})
    print(summary)

    sns.histplot(col.to_frame(), bins=50, stat="count", ax=ax)

plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

columns = [
    "aims__asset_length",
    "aims__toe_level",
    "aims__current_sop",
    "aims__design_sop",
    "aims__design_ucl",
    "aims__actual_dcl",
]

fig, axes = plt.subplots(6, 3, figsize=(12,18))

for index, column in enumerate(columns):
    ax1, ax2, ax3 = axes[index]
    ax1.set_title(column)

    col = df.get_column(column)

    skew = round(float(stats.skew(col)),3)
    kurtosis = round(float(stats.kurtosis(col)),3)
    min = round(float(col.min()),3) # type: ignore
    median = round(float(col.median()),3) # type: ignore
    max = round(float(col.max()),3) # type: ignore

    summary = pl.DataFrame({"skew": skew, "kurtosis": kurtosis, "min": min, "median": median, "max":max})
    summary = summary.transpose(include_header=True, header_name="stat", column_names=[""])

    sns.histplot(col.to_frame(), bins=50, stat="count", ax=ax1)

    ax2.table(cellText=summary.to_numpy(), loc="center")

    stats.probplot(col, dist="norm", plot=ax3)

plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

columns = [
    "aims__asset_length",
    "aims__toe_level",
    "aims__current_sop",
    "aims__design_sop",
    "aims__design_ucl",
    "aims__actual_dcl",
]

fig, axes = plt.subplots(6, 3, figsize=(12,18))

for index, column in enumerate(columns):
    ax1, ax2, ax3 = axes[index]
    ax1.set_title(column)

    col = df.get_column(column).drop_nulls()

    skew = round(float(stats.skew(col)),3)
    kurtosis = round(float(stats.kurtosis(col)),3)
    min = round(float(col.min()),3) # type: ignore
    median = round(float(col.median()),3) # type: ignore
    max = round(float(col.max()),3) # type: ignore

    summary = pl.DataFrame({"skew": skew, "kurtosis": kurtosis, "min": min, "median": median, "max":max})
    summary = summary.transpose(include_header=True, header_name="stat", column_names=[""])

    sns.histplot(col.to_frame(), bins=50, stat="count", ax=ax1)

    ax2.table(cellText=summary.to_numpy(), loc="center")

    stats.probplot(col, dist="norm", plot=ax3)

plt.tight_layout()

plt.show()

Dont think sop values are actually continuous. stands for "standard of protection".

In [ ]:
columns += ["aims__effective_cl", "aims__actual_ucl", "aims__design_dcl"]

columns.remove("aims__design_sop")
columns.remove("aims__current_sop")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

columns = [
    "aims__asset_length",
    "aims__toe_level",
    "aims__current_sop",
    "aims__design_sop",
    "aims__design_ucl",
    "aims__actual_dcl",
]

fig, axes = plt.subplots(6, 3, figsize=(12,18))

for index, column in enumerate(columns):
    ax1, ax2, ax3 = axes[index]
    ax1.set_title(column)

    col = df.get_column(column).drop_nulls()

    skew = round(float(stats.skew(col)),3)
    kurtosis = round(float(stats.kurtosis(col)),3)
    min = round(float(col.min()),3) # type: ignore
    median = round(float(col.median()),3) # type: ignore
    max = round(float(col.max()),3) # type: ignore

    summary = pl.DataFrame({"skew": skew, "kurtosis": kurtosis, "min": min, "median": median, "max":max})
    summary = summary.transpose(include_header=True, header_name="stat", column_names=[""])

    sns.histplot(col.to_frame(), bins=50, stat="count", ax=ax1)

    ax2.table(cellText=summary.to_numpy(), loc="center")

    stats.probplot(col, dist="norm", plot=ax3)

plt.tight_layout()

plt.show()

In [ ]:
crest = [
    "aims__actual_dcl", 
    "aims__actual_ucl", 
    "aims__design_dcl",
    "aims__design_ucl", 
    "aims__effective_cl", 
    "aims__toe_level"
]
corr = df[crest].drop_nulls().corr()
sns.heatmap(corr, annot=True, fmt=".5f", cmap="coolwarm", center=0)
plt.show()

In [ ]:
import numpy as np

rows = []
for c in columns:
    col = df.get_column(c).drop_nulls().to_numpy().reshape(-1, 1)
    raw_skew = float(stats.skew(col.ravel()))

    log_skew = np.nan
    log_skew = float(stats.skew(np.log1p(col.ravel())))

    # print(c)
    # print(f"skew before transform: {raw_skew}")
    # print(f"skew after transform: {log_skew}")
    # print("\n\n")

    rows.append({
        "column": c,
        "skew_raw": round(raw_skew, 3),
        "skew_log1p": None if np.isnan(log_skew) else round(log_skew, 3),
    })

transform_test = pl.DataFrame(rows)
transform_test

In [ ]:

cov = (
    df.select([
        (pl.col(column).is_not_null().mean() * 100).alias(column) for column in df.columns
    ])
    .to_pandas().T.rename(columns={0: "coverage_pct"})
    .sort_values("coverage_pct")
)
cov["coverage_pct"] = cov["coverage_pct"].round(1)

plt.figure(figsize=(9, 0.4 * len(cov)))
sns.barplot(x="coverage_pct", y=cov.index, data=cov)

plt.axvline(25, ls="--", c="grey", alpha=0.5)
plt.axvline(50, ls="--", c="grey", alpha=0.75)
plt.axvline(75, ls="--", c="grey", alpha=0.5)

plt.xlabel("coverage %")
plt.ylabel("")
plt.title("Column coverage")
plt.tight_layout()
plt.show()

cov

In [ ]:
relevent_columns = [
    "eir__condition_grade",
    "aims__asset_length",
    "aims__toe_level",
    "aims__current_sop",
    "aims__design_sop",
    "aims__design_ucl",
    "aims__actual_dcl",
    "geosure_collapsible_deposits__class",
    "geosure_compressible_ground__class",
    "geosure_landslides__class",
    "geosure_running_sand__class",
    "geosure_shrink_swell__class",
    "geosure_soluble_rocks__class",
    "aims__asset_sub_type",
    "aims__protection_type",
    "aims__primary_purpose",
    "aims__asset_maintainer",
    "aims__asset_owner",
    "aims__current_condition", 
    "_grade",
    "eir__inspection_date",
    "bedrock_geo__lex_rcs",
]




cov = (
    df.select([
        (pl.col(column).is_not_null().mean() * 100).alias(column) for column in relevent_columns
    ])
    .to_pandas().T.rename(columns={0: "coverage_pct"})
    .sort_values("coverage_pct")
)
cov["coverage_pct"] = cov["coverage_pct"].round(1)

plt.figure(figsize=(9, 0.4 * len(cov)))
sns.barplot(x="coverage_pct", y=cov.index, data=cov)

plt.axvline(25, ls="--", c="grey", alpha=0.5)
plt.axvline(50, ls="--", c="grey", alpha=0.75)
plt.axvline(75, ls="--", c="grey", alpha=0.5)

plt.xlabel("coverage %")
plt.ylabel("")
plt.title("Column coverage")
plt.tight_layout()
plt.show()

cov

In [ ]:
import polars.selectors as cs

df.select(cs.starts_with("geosure")).columns

In [ ]:
geosure_classes = [
    "geosure_collapsible_deposits__class",
    "geosure_compressible_ground__class",
    "geosure_landslides__class",
    "geosure_running_sand__class",
    "geosure_shrink_swell__class",
    "geosure_soluble_rocks__class",
]


gs_classes = df.select(geosure_classes)

for column in geosure_classes:
    value_count = df.get_column(column).value_counts(sort=True)

    unique_values = df.get_column(column).unique()

    null_pct = df.get_column(column).is_null().mean()

    print(f"column: {column}\nvalue count: {value_count}\nnull percent: {null_pct}\nunique values: {unique_values}\n")




In [ ]:
geosure_classes = [
    "geosure_collapsible_deposits__class",
    "geosure_compressible_ground__class",
    "geosure_landslides__class",
    "geosure_running_sand__class",
    "geosure_shrink_swell__class",
    "geosure_soluble_rocks__class",
]


gs_classes = df.select(geosure_classes)

rows = []

for column in geosure_classes:
    value_count = df.get_column(column).value_counts(sort=True)

    unique_values = df.get_column(column).unique()

    null_pct = df.get_column(column).is_null().mean()

    # print(f"column: {column}\nvalue count: {value_count}\nnull percent: {null_pct}\nunique values: {unique_values}\n")

    rows.append({
        "layer": column,
        "n_unique": unique_values.len(),
        "null_pct": null_pct,
    })

classes = pl.DataFrame(rows)

classes


In [ ]:

# display(pl.DataFrame([{k: v for k, v in p.items() if k != "value_counts"} for p in profile]))

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)

for ax, column in zip(axes.ravel(), geosure_classes):

    sns.countplot(
        x=df.get_column(column), 
        ax=ax, 
    )
    ax.set_title(c.replace("geosure_", "").replace("__class", ""))
    ax.set_xlabel("ordinal class")

plt.tight_layout()
plt.show()


In [ ]:
geo_str_cols = [
    column for column in df.columns
    if (column.startswith("bedrock_geo__") or column.startswith("superficial_geo__")) and df.schema[column] == pl.Utf8
]

card = (
    pl.DataFrame({
        "column": geo_str_cols,
        "n_unique": [df.get_column(col).n_unique() for col in geo_str_cols],
        "null_pct": [round(float(df.get_column(col).is_null().mean() * 100), 1) for col in geo_str_cols],
    })
    .sort("n_unique", descending=True)
)
print(card)


In [ ]:


val_counts = df.get_column("bedrock_geo__lex_rcs").value_counts(sort=True).with_columns(
    (pl.col("count").cum_sum() / pl.col("count").sum() * 100).alias("cumulative_pct")
)


threshold = int( 0.01 * len(df))


n_rare = val_counts.filter(pl.col("count").lt(threshold)).height

print(f"bedrock_geo__lex_rcs: {len(val_counts)} categories, {n_rare} below 1% ({threshold} rows) frequency")


In [ ]:

plt.figure(figsize=(15, 5))

sns.barplot(
    x=np.arange(len(val_counts)), 
    y=val_counts.get_column("count"),
)

plt.axhline(threshold, color="black", ls="--", alpha=0.75, label="1% threshold")

plt.xlabel("category rank")
plt.ylabel("count")
plt.legend()
plt.title("bedrock_geo__lex_rcs frequency long tail")
plt.tight_layout()

plt.xticks(rotation=315)

plt.show()


In [ ]:
low_card_cats = [
    "aims__asset_sub_type",
    "aims__protection_type",
    "aims__primary_purpose",
    "aims__asset_maintainer",
    "aims__asset_owner",
]



for column in low_card_cats:

    val_counts = df.get_column(column).value_counts(sort=True).with_columns(
        (pl.col("count").cum_sum() / pl.col("count").sum() * 100).alias("cumulative_pct")
    )

    n_rare = val_counts.filter(pl.col("count").lt(50)).height

    print(f"{column}: {len(val_counts)} categories, {n_rare} below 50 ocurances")
    print(val_counts.head(15))
    print()


In [ ]:

for cols in ["aims__asset_maintainer", "aims__asset_owner"]:

    norm = (
        df.select(
            pl.col(cols).alias("raw"),
            pl.col(cols).str.strip_chars().str.to_lowercase().alias("norm"),
        )
        .group_by("norm")
        .agg(pl.col("raw").unique().alias("variants"))
        .filter(pl.col("variants").list.len() > 1)
    )
    
    print(f"{cols}: {norm.height} normalised keys with >1 raw variant")
    print(norm.head(10))

In [ ]:
from pyclustertend import hopkins
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

include_cols = [
    "aims__asset_length",
    "aims__actual_dcl",
    "aims__effective_cl",
    "aims__design_ucl",
    "aims__toe_level",
    "geosure_collapsible_deposits__class",
    "geosure_compressible_ground__class",
    "geosure_landslides__class",
    "geosure_running_sand__class",
    "geosure_shrink_swell__class",
    "geosure_soluble_rocks__class",
    "aims__design_sop",
    "aims__current_sop",
]

mat = df.select(include_cols).to_numpy()
X = StandardScaler().fit_transform(SimpleImputer(strategy="most_frequent").fit_transform(mat))


np.random.seed(0)
hop_stat = 1 - hopkins(X, sampling_size=1500)   # this library has reversed stat vs r implementation

hop_stat

In [ ]:
from pyclustertend import hopkins
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

include_cols = [
    "aims__actual_dcl",
    "geosure_collapsible_deposits__class",
    "geosure_compressible_ground__class",
    "geosure_landslides__class",
    "geosure_running_sand__class",
    "geosure_shrink_swell__class",
    "geosure_soluble_rocks__class",
]

mat = df.select(include_cols).to_numpy()
X = StandardScaler().fit_transform(SimpleImputer(strategy="most_frequent").fit_transform(mat))


np.random.seed(0)
hop_stat = 1 - hopkins(X, sampling_size=1500)   # this library has reversed stat vs r implementation

hop_stat

In [ ]:
from sklearn.decomposition import PCA

include_cols = [
    "aims__asset_length",
    "aims__actual_dcl",
    "aims__effective_cl",
    "aims__design_ucl",
    "aims__toe_level",
    "geosure_collapsible_deposits__class",
    "geosure_compressible_ground__class",
    "geosure_landslides__class",
    "geosure_running_sand__class",
    "geosure_shrink_swell__class",
    "geosure_soluble_rocks__class",
    "aims__design_sop",
    "aims__current_sop",
]

mat = df.select(include_cols).to_numpy()
X = StandardScaler().fit_transform(SimpleImputer(strategy="most_frequent").fit_transform(mat))

pca = PCA().fit(X)
ev = pca.explained_variance_ratio_
cum = np.cumsum(ev)

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(np.arange(1, len(ev) + 1), ev, color="blue", alpha=0.7, label="individual")
ax.step(np.arange(1, len(cum) + 1), cum, color="black", where="mid", label="cumulative")

ax.axhline(0.9, color="grey", ls="--", lw=1)
ax.set_xlabel("component")
ax.set_ylabel("explained variance ratio")
ax.legend()

plt.tight_layout()
plt.show()

print("Components for 90% variance:", int(np.argmax(cum >= 0.9) + 1))